In [1]:
import sys
import os
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.datasets import fetch_openml
from sklearn import metrics
import cudf, cuml
%load_ext cuml.accel
import cupy as cp
%load_ext cudf.pandas
import pandas as pd
from cuml.manifold import TSNE, UMAP

In [2]:
# Set the path to the file you'd like to load
file_path = "./Data/tracks.csv"
# Load the latest version
df = pd.read_csv(file_path)

print("First 5 records:", df.head())

In [3]:
# these features are more or less what we are trying to cluster
# since these BEST describe the music
# mode and time signature may impact as well
features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", 
    #"duration_ms", "popularity",
    #"mode", "time_signature"
]

id_data = df[["name", "artists"]].copy()

X = df[features].copy()
y = df["popularity"].copy()

# convert boolean to int
X["explicit"] = df["explicit"].astype(int)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [5]:
# we are looking for tight clusters so we want low min_dist and low n_neighbors
n_neighbors = [5, 10, 20, 40]
min_dist = [0.01, 0.05, 0.1, 0.3]

possible_combinations = [(n, d) for n in n_neighbors for d in min_dist]

index = 0

In [6]:
#from umap import UMAP

# setting up reducer to be used for the rest of code
reducer = UMAP(
    n_components=2,
    n_neighbors=100,
    min_dist=0.3,
    metric="euclidean",
    random_state=42
)

embedding = reducer.fit_transform(X_scaled)
embedding.shape

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"

# just doing a basic scatter plot to see how the clusters look with the default settings
fig = px.scatter(
    x=embedding[:, 0],
    y=embedding[:, 1],
    color=y
)

fig.show()

In [ ]:
from sklearn.decomposition import PCA

# could PCA be an option for dimensionality reduction before UMAP? maybe it can help with noise reduction and speed up the process
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

embedding = reducer.fit_transform(X_pca)

In [ ]:
fig = px.scatter(
    x=embedding[:, 0],
    y=embedding[:, 1],
    #z=embedding[:, 2],
    color=y
)

fig.show()
# PCA is losing some structure. UMAP is doing a better job here
# Some research papers were using PCA before K means, but others skipped PCA and went straight to UMAP.
# UMAP is probably ther better option here
# Interestingly though, we can see how the left cluster is the most popular songs

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    max_iter=1500,
    metric="euclidean",
    random_state=42
)

embedding = tsne.fit_transform(X_scaled)

import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=15)
labels = clusterer.fit_predict(embedding)

labels_plot = labels.copy()
labels_plot = labels_plot.astype(str)
labels_plot[labels == -1] = "noise"

gray_out_noise = {"noise": "lightgray"}

fig = px.scatter(
        x=embedding[:, 0],
        y=embedding[:, 1],
        color=labels_plot,
        color_discrete_map=gray_out_noise,
        hover_data={
            "name": id_data["name"],
            "artists": id_data["artists"],
            "popularity": y
        }
    )

fig.show()
# Just want to test TSNE here. This looks worse than PCA before

In [ ]:
from Functions.reducers import reduce

embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

#this is just bad

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
embedding, fig = reduce(X_scaled, y, possible_combinations[index], id_data)
fig.show()

index += 1

In [ ]:
good_combinations = [possible_combinations[12], possible_combinations[13], possible_combinations[14]]

In [ ]:
n_neighbors, min_dist = possible_combinations[12]
reducer = UMAP(
    n_components=2,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    metric="euclidean",
    random_state=42
)

embedding = reducer.fit_transform(X)

fig = px.scatter(
    x=embedding[:, 0],
    y=embedding[:, 1],
    color=y,
    color_continuous_scale="Viridis",
    hover_data={
        "name": id_data["name"],
        "artists": id_data["artists"],
        "popularity": y
    }
)

fig.update_layout(
    title=f"UMAP with n_neighbors={n_neighbors} and min_dist={min_dist}",
    xaxis_title="UMAP Dimension 1",
    yaxis_title="UMAP Dimension 2",
    legend_title="Popularity"
)

fig.show()

In [ ]:
n_neighbors, min_dist = possible_combinations[13]
reducer = UMAP(
    n_components=2,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    metric="euclidean",
    random_state=42
)

embedding = reducer.fit_transform(X)

fig = px.scatter(
    x=embedding[:, 0],
    y=embedding[:, 1],
    color=y,
    color_continuous_scale="Viridis",
    hover_data={
        "name": id_data["name"],
        "artists": id_data["artists"],
        "popularity": y
    }
)

fig.update_layout(
    title=f"UMAP with n_neighbors={n_neighbors} and min_dist={min_dist}",
    xaxis_title="UMAP Dimension 1",
    yaxis_title="UMAP Dimension 2",
    legend_title="Popularity"
)

fig.show()

In [ ]:
n_neighbors, min_dist = possible_combinations[14]
reducer = UMAP(
    n_components=2,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    metric="euclidean",
    random_state=42
)

embedding = reducer.fit_transform(X)

fig = px.scatter(
    x=embedding[:, 0],
    y=embedding[:, 1],
    color=y,
    color_continuous_scale="Viridis",
    hover_data={
        "name": id_data["name"],
        "artists": id_data["artists"],
        "popularity": y
    }
)

fig.update_layout(
    title=f"UMAP with n_neighbors={n_neighbors} and min_dist={min_dist}",
    xaxis_title="UMAP Dimension 1",
    yaxis_title="UMAP Dimension 2",
    legend_title="Popularity"
)

fig.show()